# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [10]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from LLMClient import get_client, get_models
from openai import OpenAI


In [11]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OLLAMA_API_KEY')

if api_key and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = get_models()["cheap"]
config = get_client("ollama")
openai = config["client"]

API key looks good so far


In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [27]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
And wait, take another look at the JSON response and make sure it is correctly formatted.
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [15]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [35]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [18]:
select_relevant_links("https://edwarddonner.com")

{
    "links": [
        {"type": "home page", "url": "https://edwarddonner.com/"},
        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
        {"type": "project page", "url": "https://edwarddonner.com/avatar/"},
        {"type": "project page", "url": "https://edwarddonner.com/proficient/"},
        {"type": "project page", "url": "https://edwarddonner.com/connect-four/"},
        {"type": "project page", "url": "https://edwarddonner.com/outsmart/"},
        {"type": "blog page", "url": "https://edwarddonner.com/posts/"},
        {"type": "social profile", "url": "https://www.linkedin.com/in/eddonner/"},
        {"type": "social profile", "url": "https://twitter.com/edwarddonner"},
        {"type": "social profile", "url": "https://www.facebook.com/edward.donner.52"}
    ]
}


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [34]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [23]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-oss:20b
{
  "links": [
    {
      "type": "home page",
      "url": "https://edwarddonner.com/"
    },
    {
      "type": "about page",
      "url": "https://edwarddonner.com/about-me-and-about-nebula/"
    },
    {
      "type": "curriculum page",
      "url": "https://edwarddonner.com/curriculum/"
    },
    {
      "type": "linkedin",
      "url": "https://www.linkedin.com/in/eddonner/"
    },
    {
      "type": "twitter",
      "url": "https://twitter.com/edwarddonner"
    },
    {
      "type": "facebook",
      "url": "https://www.facebook.com/edward.donner.52"
    }
  ]
}
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [24]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b
{
    "links": [
        {"type": "home", "url": "https://huggingface.co/"},
        {"type": "enterprise overview", "url": "https://huggingface.co/enterprise"},
        {"type": "pricing", "url": "https://huggingface.co/pricing"},
        {"type": "blog", "url": "https://huggingface.co/blog"},
        {"type": "learning resources", "url": "https://huggingface.co/learn"},
        {"type": "careers page", "url": "https://apply.workable.com/huggingface/"},
        {"type": "github", "url": "https://github.com/huggingface"},
        {"type": "twitter", "url": "https://twitter.com/huggingface"},
        {"type": "linkedin", "url": "https://www.linkedin.com/company/huggingface/"}
    ]
}
Found 9 relevant links


{'links': [{'type': 'home', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise overview', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [25]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b
{
  "links": [
    {"type":"home page","url":"https://huggingface.co/"},
    {"type":"models page","url":"https://huggingface.co/models"},
    {"type":"datasets page","url":"https://huggingface.co/datasets"},
    {"type":"spaces page","url":"https://huggingface.co/spaces"},
    {"type":"enterprise page","url":"https://huggingface.co/enterprise"},
    {"type":"pricing page","url":"https://huggingface.co/pricing"},
    {"type":"blog","url":"https://huggingface.co/blog"},
    {"type":"learn resources","url":"https://huggingface.co/learn"},
    {"type":"documentation","url":"https://huggingface.co/docs"},
    {"type":"careers page","url":"https://apply.workable.com/huggingface/"},
    {"type":"community forum","url":"https://discuss.huggingface.co/"},
    {"type":"github repository","url":"https://github.com/huggingface"},
    {"type":"twitter","url":"https://twitter.com/huggingface"},
    {"type":"linkedin","url":"

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [30]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b
{
    "links": [
        {"type": "main page", "url": "https://huggingface.co/"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "pricing page", "url": "https://huggingface.co/pricing"},
        {"type": "careers page", "url": "https://apply.workable.com/huggingface/"},
        {"type": "blog page", "url": "https://huggingface.co/blog"},
        {"type": "github page", "url": "https://github.com/huggingface"},
        {"type": "twitter page", "url": "https://twitter.com/huggingface"},
        {"type": "linkedin page", "url": "https://www.linkedin.com/company/huggingface/"},
        {"type": "community forum", "url": "https://discuss.huggingface.co/"},
        {"type": "discord invite", "url": "https://huggingface.co/join/discord"},
        {"type": "support page", "url": "https://huggingface.co/support"}
    ]
}
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n3 days ago\n•\n2.59M\n•\n3.19k\npoolside/Laguna-S-2.1\nUpdated\n2 days ago\n•\n56.4k\n

In [32]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [33]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss:20b
{
    "links": [
        {"type": "homepage", "url": "https://huggingface.co"},
        {"type": "enterprise page", "url": "https://huggingface.co/enterprise"},
        {"type": "pricing page", "url": "https://huggingface.co/pricing"},
        {"type": "careers page", "url": "https://apply.workable.com/huggingface/"},
        {"type": "documentation page", "url": "https://huggingface.co/docs"},
        {"type": "blog page", "url": "https://huggingface.co/blog"},
        {"type": "GitHub page", "url": "https://github.com/huggingface"},
        {"type": "LinkedIn page", "url": "https://www.linkedin.com/company/huggingface/"},
        {"type": "Twitter page", "url": "https://twitter.com/huggingface"}
    ]
}
Found 9 relevant links


# Hugging Face – The AI Community Building the Future  

## 🚀 About Us  
Hugging Face is the **platform where the global machine‑learning community builds, shares, and deploys AI models, datasets, and applications**. Since 2016, we have grown into a vibrant ecosystem that merges research, engineering, and open‑source collaboration to accelerate AI innovation.

| Feature | What it means |
|---------|---------------|
| **2 + million models** | Thousands of ready‑to‑use models for language, vision, audio, and multimodal tasks. |
| **500 k+ datasets** | Curated, high‑quality data spanning every domain – from text corpora to bio‑sequences. |
| **1 + million applications** | End‑to‑end AI apps built with our “Spaces” – team‑owned, shareable, and instantly deployable. |

---

## 🌍 Vision & Culture  
- **Community‑first**: We are *not* just a software company; we’re a network of researchers, developers, and hobbyists who iterate together.  
- **Open‑source heart**: Every model, dataset, and code library is freely available. Contributions are celebrated and people gain real recognition through collaborations.  
- **Trust & transparency**: Ethical AI is built into our data curation, model governance, and usage policies.  
- **Hands‑on learning**: From tutorials to live‑coding Jupyter notebooks, we make ML approachable for anyone.  

---

## ✨ Products & Offerings  

| Category | What you get |
|----------|--------------|
| **Models** | Plug‑and‑play large language models, vision encoders, speech recognizers, and more. |
| **Datasets** | Cleaned, licensed, pre‑downloaded datasets ready for training and evaluation. |
| **Spaces** | In‑browser environments (React, Streamlit, Gradio) to prototype, demo, and share AI applications in minutes. |
| **Buckets** | Scalable cloud storage for model weights, datasets, and endpoint checkpoints. |
| **Inference Pipelines** | Endpoints and providers that let you run models at scale with minimal latency. |
| **HuggingChat** | Conversational AI built on top of the same model hub, ideal for building chat assistants. |
| **Enterprise Plans** | Hugging Face PRO, dedicated support, private model hosting, auditing, and policy compliance. |

---

## 🤝 Who Uses Hugging Face?  

| Use‑case | Customers |
|----------|-----------|
| **Startups & Indie Developers** | Rapid prototyping, MVPs, open‑source innovation. |
| **Large Enterprises** | Secure, private model hosting, internal AI pipelines for finance, healthcare, and retail. |
| **Academic Research** | Access to cutting‑edge models for reproducible experiments. |
| **Product Teams** | Full‑stack AI features in mobile, web, and edge devices (via WebGPU, ONNX, TensorRT). |

---

## 💼 Careers & Culture  

| Role | What we look for |
|------|-----------------|
| **Machine‑Learning Engineer** | Experience with deep learning frameworks (PyTorch/TensorFlow), model optimization, and inference deployment. |
| **Data Scientist** | Strong statistical background, dataset creation, and experimentation. |
| **Open‑Source Contributor** | Passion for building and maintaining public projects; strong communication and community skills. |
| **Product & Design** | Building usable AI tools; user‑centric design thinking. |
| **Infrastructure & Ops** | Cloud scaling, security, and reliability engineering. |

**Why Join Us?**  
- *Impact*: Spearhead projects that reach millions of developers worldwide.  
- *Community*: An open culture where ideas are freely shared and intellectual property is open‑source.  
- *Learning*: Access to the latest research, workshops, and data science conferences.  
- *Growth*: Clear career paths across engineering, research, and product leadership.  

---

## 🔗 Get Involved  

- **Explore Models & Datasets** – 2 + million ready‑to‑use assets.  
- **Build an App** – Create a Space in minutes and ship to the community.  
- **Join the Community** – Discord, Forums, GitHub, and the Hugging Face Blog.  
- **Partner with Enterprise** – Private hosting, SLAs, and compliance.  

---

## 📈 Invest in the Future  

Hugging Face’s open‑source, community‑driven model hub is driving the next wave of AI adoption. By combining massive scale (2M+ models, 500k+ datasets) with flexible enterprise-grade services, we empower every organization—small or large—to innovate quickly, safely, and responsibly.  

> **“The future is collaborative, and Hugging Face is the infrastructure that makes it happen.”**  

--- 

Explore more at [huggingface.co](https://huggingface.co) and join the conversation on Discord or GitHub.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [38]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [39]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face – The AI Community Building the Future

---

## About Us  
Hugging Face is a globally‑recognized hub for the machine‑learning community, where developers, researchers, and businesses collaborate on open‑source models, datasets, and end‑to‑end AI applications. We power conversations around the world with tools that let anyone share, discover, and build on top of 2 + M models and 500 k+ datasets.

- **Mission**: Democratize AI by making state‑of‑the‑art models and data openly available.  
- **Vision**: A future where every innovation in machine learning is a joint, community‑driven effort.  
- **Community‑first**: A vibrant mix of GitHub, Discord, forum discussions, and blog posts where ideas spark new breakthroughs.

---

## What We Offer

| Category | Highlights | Use Cases |
|----------|------------|-----------|
| **Models** | 2 + M models ranging from OCR and vision to large‑language models (e.g., Solar‑Open2‑250B, Qwen‑3.6‑27B). | Text generation, translation, image captioning, code synthesis, etc. |
| **Datasets** | 500 k+ datasets: code corpora, reasoning collections, industrial corpora. | Fine‑tuning, research benchmarks, data‑driven product engineering. |
| **Spaces** | Interactive demos and apps: video generation from text, WebGPU‑based 27B LLMs, real‑time voice chat. | Rapid prototyping, UI/UX experience for ML models. |
| **Buckets** | Secure storage for model artefacts and training data, integrated with inference endpoints. | Continuous training pipelines, data governance. |
| **Enterprise** | PRO & Enterprise Support, dedicated inference providers, custom endpoints, storage buckets. | Ad‑tech, fintech, retail, and healthcare sectors demanding production‑grade AI. |

---

## Community

- **Open‑Source First**: All models, datasets and code are public on GitHub; anyone can contribute or fork.  
- **Collaborative Culture**: Baidu/Unlimited‑OCR, upstage/Solar‑Open2‑250B, and Qwen‑Image‑Edit LoRAs are example projects fueled by community contributions.  
- **Events & Knowledge Sharing**: Daily papers, blog posts, Discord channels, and a forum keep the conversation alive and deliver daily insights into the cutting‑edge of AI research.

---

## Customers & Partners

While specific client names are not disclosed publicly, Hugging Face’s platform is adopted by:

- Fortune 500 enterprises seeking scalable inference services.  
- Academic labs and research institutions developing new NLU/NLP benchmarks.  
- Start‑ups building AI‑driven products in e‑commerce, media, and health tech.  

Key features such as **Enterprise Support** and **Inference Endpoints** ensure that critical AI workloads run reliably in production.

---

## Careers

Hugging Face is expanding its world‑class team across **engineering, product, research, data science, and community operations**. The culture fosters collaboration and encourages:

- **Open‑source contributions** as part of professional growth.  
- **Cross‑functional teamwork** to bring models from research to everyday products.  
- **Continual learning** through internal workshops and mentorship.

> *If you’re passionate about building open‑source AI tools that accelerate innovation, join a community that believes the best code and data are shared and improve collectively.*

---

## Get Involved

- **Explore**: Browse ★ 2 M+ models & 500 k+ datasets on the main site.  
- **Build**: Create a *Space* to demo your model in minutes.  
- **Collaborate**: Contribute to public repositories or join our vibrant Discord community.  
- **Partner**: Check out our Enterprise plans to integrate Hugging Face into your business stack.

---

*Hugging Face – Where AI ideas become worldwide realities.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>